<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB15_Case_Study_CLIWOC_Nationality_from_Ship_Routes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB15 · Class 15 — Case Study: CLIWOC Historical Ship Logbooks, Classifying Nationality from Routes**

## Block 4: Proyectos — Case Studies (continued)

`NB14` used real 21st-century weather data. This case study reaches back much further: real 18th-century ship logbooks, digitized for climate research, used here to ask a different kind of question — can a ship's **route and timing alone** reveal which nation it sailed for?

**The real-world problem**: across the mid-to-late 1700s, the British, Dutch, Spanish, and French each ran established maritime trade networks tied to their colonial possessions — Spain's transatlantic and Manila galleon routes, the Dutch VOC's Cape route to the East Indies, British East India Company and Atlantic trade, French Caribbean and Indian Ocean trade. If those networks were real and geographically distinct, a model should be able to recover "which nation" from nothing but *where* and *when* a ship was — a genuine test of whether real historical trade geography is learnable from data, not an arbitrary classroom label.

We download the real dataset live from Kaggle, so the exact rows and years available may vary slightly depending on the current release — this notebook is deliberately written to **discover** the real data's structure and time span rather than assume fixed numbers, exactly the habit `NB02`'s "load → inspect" workflow was built around.

Following `NB14`'s corrected structure, this class again has **two parts**: **Part A** (Sections 8–9) validates a modeling approach with a standard random split; **Part B** (Section 10) is the real test — training only on the earlier ~80% of available years and predicting nationality for the **later years the model has never seen**, checking whether these trade-route patterns actually held stable across time, or shifted.

### Learning objectives

By the end of this class, students will be able to:
- Explain why "route → nationality" is a real, historically grounded question, not an arbitrary label.
- Download a real dataset from Kaggle using secure, session-only credentials.
- Work with a real, messy historical dataset — including discovering its actual structure and class balance, rather than assuming them in advance.
- Visualize class-separated geographic data on a real map and use it to sanity-check a modeling premise before training anything.
- Handle class imbalance with a naive baseline and `class_weight="balanced"`.
- Design a temporal holdout that matches the real question being asked (generalizing across years, not being told the "answer" it already saw).

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap, today's roadmap | 5 min | Theory |
| 2 | What is CLIWOC, and why "route → nationality" is a real question | 15 min | Theory |
| 3 | Setting up Kaggle API access | 10 min | Practice |
| 4 | Downloading the real dataset | 10 min | Practice |
| 5 | Exploring the data: columns, nationalities, routes on a real map | 20 min | Practice |
| 6 | Preparing features and confronting real class imbalance | 10 min | Theory + Practice |
| 7 | Applying `NB13`'s decision framework | 5 min | Theory + Practice |
| 8 | Part A: training and comparing models (methodology validation) | 15 min | Practice |
| 9 | Part A: evaluation | 10 min | Practice |
| 10 | Part B: a genuine test — forecasting held-out years | 10 min | Practice |
| 11 | Interpreting Part B, and connecting it to real history | 5 min | Practice |
| 12 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

- **`NB14`**: real ECMWF weather data, a genuine held-out-year forecast, and the lesson that a fair generalization test has to hold out the *right* variable (a year, not a season, when the target is seasonal).
- **`NB15`** (today): a different kind of real data — historical, sparse, imbalanced — and the same discipline applied to a temporal holdout that matches *this* problem's real question.

---

## 2. What is CLIWOC, and why "route → nationality" is a real question

**[CLIWOC](https://en.wikipedia.org/wiki/CLIWOC)** (Climatological Database for the World's Oceans) was a real research project that converted historical ships' logbooks — British, Dutch, French, and Spanish, 1750–1850 — into a standardized digital database, originally to reconstruct historical climate and wind patterns from centuries of daily noon observations. That means every row in this dataset is a **real entry a real ship's officer wrote down** at sea, up to 275 years ago.

Today's question uses the same data for a different purpose: each of the four nations ran distinct, real trade networks shaped by their colonial territories and monopolies — Spain's transatlantic and Pacific galleon routes, the Dutch East India Company's Cape-of-Good-Hope route to Indonesia, British Atlantic and Indian Ocean trade, French Caribbean and Indian Ocean trade. If those networks were geographically real and distinct (which real maritime history says they were), a model trained only on **where** and **when** a logbook entry was recorded should be able to recover **which nation** wrote it — genuine historical geography, learnable from data, not an arbitrary label invented for a homework problem.

---

## 3. Setting up Kaggle API access

The full CLIWOC database — over 287,000 real logbook entries, 180 columns, spanning 1662–1855 — is published on Kaggle. Downloading it needs a free Kaggle account and API key:

1. Go to [kaggle.com/settings](https://www.kaggle.com/settings), scroll to **API**, and click **Create New Token** — this downloads a `kaggle.json` file containing your username and key.
2. Open that file (any text editor) to read your username and key values.
3. Run the cell below and paste each one when prompted.

Just like `NB14`'s CDS key, we use `getpass` so your credentials are **never written into this notebook's saved code or output** — unlike uploading a `kaggle.json` file directly into a shared notebook, which is exactly how this repository leaked a real Kaggle key earlier in this course's history.

In [ ]:
%pip install -q kaggle

import os
from getpass import getpass

KAGGLE_USERNAME = getpass("Your Kaggle username: ").strip()
KAGGLE_KEY = getpass("Your Kaggle API key (from the downloaded kaggle.json): ").strip()

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")
with open(kaggle_json_path, "w") as f:
    f.write('{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
os.chmod(kaggle_json_path, 0o600)

print("Kaggle credentials saved for this session (not stored anywhere in this notebook).")

---

## 4. Downloading the real dataset

In [ ]:
!kaggle datasets download -d cwiloc/climate-data-from-ocean-ships --force --unzip

import os

csv_files = [f for f in os.listdir(".") if f.lower().endswith(".csv")]
print("CSV files in the downloaded archive:", csv_files)

---

## 5. Exploring the data: columns, nationalities, routes on a real map

`NB02`'s starting questions, on real 18th-century data this time. The archive's main file is normally `CLIWOC15.csv` — load it and look at its real columns first, exactly like `NB02`'s "load → inspect" habit, since with 180 real columns there's no point guessing:

In [ ]:
import pandas as pd

raw = pd.read_csv("CLIWOC15.csv", low_memory=False)
print(raw.shape)
raw.columns.tolist()

The columns worth today's four-nation route question: `Lon3`/`Lat3` (a cleaned decimal position, among several position representations in this dataset), `Year`, and `Nationality`. Keep only rows where all four are present, and restrict to the four well-represented navies this class is about — matching on the values case-insensitively, since we haven't confirmed their exact casing in this release yet:

In [ ]:
print(raw["Nationality"].value_counts())

Filter to real, complete rows for our four target nations, and rename the position columns to plain `longitude`/`latitude` so the rest of the notebook doesn't need to know the original column names:

In [ ]:
target_nations = ["BRITISH", "DUTCH", "SPANISH", "FRENCH"]
nat_upper = raw["Nationality"].astype(str).str.upper().str.strip()

cliwoc = raw.loc[nat_upper.isin(target_nations), ["Lon3", "Lat3", "Year", "Nationality"]].copy()
cliwoc["Nationality"] = nat_upper[nat_upper.isin(target_nations)]
cliwoc = cliwoc.rename(columns={"Lon3": "longitude", "Lat3": "latitude"}).dropna()

print(cliwoc.shape)
cliwoc.head()

Now the real, discovered summary — no assumptions, just what this download actually contains:

In [ ]:
print(cliwoc["Nationality"].value_counts())
print()
print("Year range:", cliwoc["Year"].min(), "-", cliwoc["Year"].max())

**Read your own output**: are the four classes close to balanced, or is one nation noticeably rarer than the others? Real archive coverage (how many logbooks from each nation survived and were digitized) rarely produces a perfectly balanced dataset — a real-world imbalance worth carrying forward, in the same spirit `NB09` flagged for its own real, self-derived labels.

Before training anything, test the actual premise visually: does each nation's real logbook data trace a geographically distinct pattern?

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

sample = cliwoc.sample(min(5000, len(cliwoc)), random_state=42)
colors = {"BRITISH": "tab:blue", "DUTCH": "tab:orange", "SPANISH": "tab:green", "FRENCH": "tab:red"}

fig = plt.figure(figsize=(12, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(resolution="110m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.add_feature(cfeature.LAND, facecolor="whitesmoke")

for nation, color in colors.items():
    subset = sample[sample["Nationality"] == nation]
    ax.scatter(subset["longitude"], subset["latitude"], s=4, alpha=0.5,
               label=nation, color=color, transform=ccrs.PlateCarree())

ax.legend(markerscale=3, loc="lower left")
year_lo, year_hi = int(cliwoc["Year"].min()), int(cliwoc["Year"].max())
ax.set_title(f"Real CLIWOC logbook positions by nationality, {year_lo}-{year_hi} ({len(sample):,}-entry sample)")
plt.show()

**Read your own map**: can you see the Spanish transatlantic/Pacific routes, the Dutch Cape-of-Good-Hope corridor toward Indonesia, the British Atlantic and Indian Ocean presence? If these four colors separate into visually distinct regions, that is real, direct evidence the "route reveals nationality" premise holds *before* we ever train a model — the map itself is doing genuine exploratory data analysis, `NB02`-style, on a real historical question.

---

## 6. Preparing features and confronting real class imbalance

Features: `latitude`, `longitude` (where) and `Year` (when). Target: `Nationality`. (`CLIWOC15.csv` doesn't reliably carry a clean `Month` column across all entries the way `Year` does, so we keep the feature set to what we've confirmed is complete.)

In [ ]:
feature_cols = ["latitude", "longitude", "Year"]
X = cliwoc[feature_cols]
y = cliwoc["Nationality"]

print(y.value_counts(normalize=True).round(3))

**Read the cell above's output**: whichever nation came out rarest, a model that ignores it entirely could still score high overall accuracy — `NB03`'s original imbalance warning, now with real numbers behind it. We'll use two concrete tools against this: a **naive baseline** to know what "cheating by ignoring the minority class" would actually score, and scikit-learn's `class_weight="balanced"` option, which up-weights minority-class errors during training instead of treating every mistake equally.

---

## 7. Applying `NB13`'s decision framework

1. **Labels?** Yes — `Nationality` is real and known for every entry.
2. **Data shape?** Tabular — three simple numeric features per row.
3. **Data volume?** Look at `X.shape[0]` above — likely tens of thousands of rows or more, comfortably in the range where either classical ML or a neural network could work; `NB13`'s framework doesn't strongly favor one here the way it did for `NB06`'s 308-row yacht data.

Given the framework doesn't push hard in either direction this time, we'll use a classical ensemble (fast, interpretable, easy to weight for imbalance) — but note for yourself that this is a case where trying a small MLP (`NB07` style) as homework is a genuinely open question, not a foregone conclusion.

---

## 8. Part A: training and comparing models (methodology validation)

A random, stratified split first — the same kind of "validate the approach" step as `NB14`'s Part A:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

Compare the naive baseline against a class-weighted Random Forest:

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_scaled, y_train)
print("Naive baseline (always predict the majority class) accuracy:", round(baseline.score(X_test_scaled, y_test), 3))

rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf.fit(X_train_scaled, y_train)
print("Random Forest accuracy:", round(rf.score(X_test_scaled, y_test), 3))

---

## 9. Part A: evaluation

Overall accuracy hides how each individual nation is doing — a full report matters more here than usual, given the imbalance:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = rf.predict(X_test_scaled)
print(confusion_matrix(y_test, y_pred, labels=rf.classes_))
print()
print(classification_report(y_test, y_pred, labels=rf.classes_))

**Read your own report**: is the rarest nation's recall noticeably lower than the other three, even with `class_weight="balanced"`? That would be an honest, expected finding given how little data exists for it — a real limitation of the underlying archive, not a modeling mistake to fix away.

---

## 10. Part B: a genuine test — forecasting held-out years

Part A's random split mixes entries from every era into both train and test — a fair methodology check, but not a real test of whether these route patterns *held up over time*. The genuine question: train only on the earlier years, then predict nationality for **the most recent years the model has never seen**, exactly the same discipline `NB14` used for its held-out year, applied to the variable that actually matters for *this* question (time period, not season).

Since we downloaded this dataset live and didn't assume its year range in advance, the split point is computed from the real data below — roughly an 80/20 split by distinct year, not by row count, so a handful of very well-documented years can't quietly dominate the cutoff:

In [ ]:
distinct_years = sorted(cliwoc["Year"].unique())
cutoff_year = distinct_years[int(len(distinct_years) * 0.8)]

train_era = cliwoc[cliwoc["Year"] < cutoff_year]
test_era = cliwoc[cliwoc["Year"] >= cutoff_year]

X_train_era, y_train_era = train_era[feature_cols], train_era["Nationality"]
X_test_era, y_test_era = test_era[feature_cols], test_era["Nationality"]

print(f"Held-out cutoff year: {cutoff_year}")
print(f"Train (years {distinct_years[0]}-{cutoff_year - 1}):", X_train_era.shape)
print(f"Test  (years {cutoff_year}-{distinct_years[-1]}):", X_test_era.shape)
print(y_test_era.value_counts(normalize=True).round(3))

Train a fresh model — this must **not** reuse the Part A model, which already saw some held-out-era rows in its own training split:

In [ ]:
scaler_era = StandardScaler()
X_train_era_scaled = scaler_era.fit_transform(X_train_era)
X_test_era_scaled = scaler_era.transform(X_test_era)

rf_era = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf_era.fit(X_train_era_scaled, y_train_era)

y_pred_era = rf_era.predict(X_test_era_scaled)
print(classification_report(y_test_era, y_pred_era, labels=rf_era.classes_))

---

## 11. Interpreting Part B, and connecting it to real history

**Compare this report to Part A's.** A meaningful drop here would be a genuinely interesting historical finding, not a failure: it would suggest trade routes in the held-out years shifted from the pattern the model learned on earlier years.

Look up the printed `cutoff_year` from the cell above, and check what was happening historically around it for the nations in this dataset — the late 18th century includes real, disruptive events such as the American Revolutionary War (1775–1783, reshaping British Atlantic trade) and the French Revolution (from 1789, disrupting French shipping soon after). If your computed cutoff falls near one of these, that is a plausible real explanation for any accuracy drop you see; if it doesn't, the honest move is to say so and look for a different explanation rather than reach for the nearest famous date.

This is exactly the honest use of a genuine holdout: it doesn't just score a model, it can **surface a real historical question worth investigating further** (was there a real shift, and if so, in which nation's routes specifically?) — a hypothesis this notebook raises but does not claim to prove; you would need real historical analysis, not just one accuracy number, to confirm it.

---

## Class summary

- CLIWOC turns 18th-century ship logbooks into real, structured data — real observations, real class imbalance, real historical stakes behind the numbers, downloaded live from Kaggle with secure, session-only credentials.
- "Route reveals nationality" is a genuine historical claim, checked visually on a real map before any model touched the data.
- A naive baseline and `class_weight="balanced"` are two concrete, complementary tools against real class imbalance — used together, not as a substitute for reading the full per-class report.
- Part A (random split) validates a modeling approach; Part B (train on earlier years, test on a later held-out era) is the real test — matching the holdout variable to the actual question, exactly as `NB14` established.
- A real accuracy drop across a genuine holdout isn't a failure to explain away — it's a legitimate signal worth connecting to real history.

## For the next class

Another Block 4 case study, using real terrain/elevation data — continuing the same pattern: a real dataset, a real question, and a genuine (not just methodological) holdout wherever the question calls for one.

## Homework / Practice Ideas

1. Add `ShipType` (from the raw dataset, encoded) as a fourth feature — does it improve Part A's per-class recall, especially for the rarest nation?
2. Try a small MLP (`NB07` style) on this task, per Section 7's open question — does it beat the Random Forest here, given how much more data this dataset has than most of this course's other classification tasks?
3. Narrow Part B's test set to just the two or three years right after `cutoff_year` instead of the full held-out era — does recall for any one nation drop specifically sharply right around that boundary?
4. Recompute Part A using `class_weight=None` (the default, unweighted) instead of `"balanced"` — how much does the rarest nation's recall change, and does overall accuracy go up or down?
5. Using the map from Section 5, pick one nation and describe (in a markdown cell) what its real historical trade routes should look like — does the plotted data match your own historical expectation?

> ***As always: a real historical dataset can teach you as much about history as about machine learning — Section 11's interpretation is not optional decoration, it's the actual point of using data this old.***